In [ ]:
import pathlib
import util
import polars

In [ ]:
gdf = util.generate_geometry_table_from_shapefile(pathlib.Path("OffshoreRenewable_Energy_Infrastructure_Regions.zip"))
with polars.Config(
    tbl_rows=16
):
    display(gdf["Name"].unique().sort())

gdf = (
    gdf
    .filter(
        polars.col("Status").eq("Declared"),
    )
    .select(
        polars.concat_str(
            polars.col("Region"),
            separator=":",
        ).alias("tag"),
        polars.col("geometry"),
    )
)
gdf

In [ ]:
h3_indexed_df = util.generate_wkt_to_h3_index(
    gdf=gdf,
    h3_resolution=7,
)

In [ ]:
import pydeck
import polars_h3

# Aggregate and generate h3 layers
h3_layers = util.generate_pydeck_hexagon_layers(
    df=(
        # H3 Aggregated dataframe with dataset and number of records attributes
        h3_indexed_df
    ),
    hexagon_index_column_name="h3_index",
    aggregate_column_name="nRecords",
    n_quantiles = 10,
    color_palette="plasma",
)

# Run the map
deck = pydeck.Deck(
    layers=h3_layers,
    map_style=pydeck.map_styles.LIGHT_NO_LABELS,
    # tooltip=util.TOOLTIP,
    initial_view_state=util.generate_view_state(
        df=h3_indexed_df.with_columns(
            polars_h3.cell_to_lat(polars.col("h3_index")).alias("decimalLatitude"),
            polars_h3.cell_to_lng(polars.col("h3_index")).alias("decimalLongitude"),
        ),
    ),
)
deck.show()